In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
REGISTRY   = "docker.io/USERNAME"  # replace with your registry
IMAGE_NAME = "csgo-train"
TAG        = "latest"
NAMESPACE  = "default"
JOB_NAME   = "csgo-train"
IMAGE      = f"{REGISTRY}/{IMAGE_NAME}:{TAG}"

In [ ]:
# ── Local training baseline (skip if submitting to Kubernetes) ─────────────────
from pathlib import Path
import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression

DATA_DIR   = Path("../data")
MODEL_PATH = DATA_DIR / "model.pkl"

X_train = pd.read_csv(DATA_DIR / "X_train.csv", index_col=0)
y_train = pd.read_csv(DATA_DIR / "y_train.csv", index_col=0).squeeze()

model = LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs")
model.fit(X_train, y_train)

train_acc = model.score(X_train, y_train)
joblib.dump(model, MODEL_PATH)

print(f"Train accuracy : {train_acc:.4f}")
print(f"Model saved    : {MODEL_PATH}")

In [ ]:
# ── Build + push Docker image ─────────────────────────────────────────────────
import subprocess

def sh(cmd: str) -> str:
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    return result.stdout

sh(f"docker build -t {IMAGE} .")
sh(f"docker push {IMAGE}")
print(f"Pushed {IMAGE}")

In [ ]:
# ── Submit Kubernetes Job ─────────────────────────────────────────────────────
sh(f"kubectl delete job {JOB_NAME} -n {NAMESPACE} --ignore-not-found")
sh(f"kubectl apply -f job.yaml -n {NAMESPACE}")
sh(f"kubectl get job {JOB_NAME} -n {NAMESPACE}")

In [ ]:
# ── Stream logs (blocks until job finishes) ───────────────────────────────────
sh(
    f"kubectl wait pod -l job-name={JOB_NAME} -n {NAMESPACE} "
    f"--for=condition=Ready --timeout=120s"
)
sh(f"kubectl logs -f job/{JOB_NAME} -n {NAMESPACE}")
sh(f"kubectl get job {JOB_NAME} -n {NAMESPACE}")